In [0]:
import sys
import os
from pyspark.sql import functions as F

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)


from src.fetch_tomtom import fetch_tomtom
from src.fetch_openmeteo import fetch_openmeteo
from src.fetch_openaq_measurements import fetch_openaq_measurements

locations_path = "/Volumes/bg_traffic/bg_traffic_bronze/landing/openaq_locations"

latest_locations_file = spark\
    .read.format("binaryFile").load(locations_path).select("path","modificationTime")\
        .orderBy(F.col("modificationTime").desc()).first()

if latest_locations_file is None:
    raise Exception("Nije pronadjen nijedan OpenAQ locations fajl")

latest_locations_path = latest_locations_file["path"]

print(f"Poslednji OpenAQ fajl: {latest_locations_path} ")

locations_df = spark.read.option("multiLine",True).json(latest_locations_path)

locations_id = locations_df.select(F.explode("data.results").alias("location"))\
    .select(F.col("location.id").alias("location_id"))\
        .where(F.col("location_id").isNotNull()).distinct().collect()




print("Preuzimanje podataka u Volume -> pokrecem fetch_tomtom fetch_openmeteo funkcije")
fetch_tomtom()
fetch_openmeteo()

for row in locations_id:
    location_id = row["location_id"]
    print(f"Preuzimam measurements za lokaciju:{location_id}")
    fetch_openaq_measurements(location_id)

